<a href="https://colab.research.google.com/github/Armando122104/Proyecto_Unidad_Datos/blob/main/Normalizacion_Modificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests
import io

PROCESO DE NORMALIZACION

Tabla 1

In [3]:
# TABLA 1 PARA NORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MSeasons.csv"
response = requests.get(url)
response.raise_for_status()
m_seasons = pd.read_csv(io.StringIO(response.text))

# Datos desnormalizados, solo los primeros 10 para la demostracion
print(m_seasons.head());
print("/");

# --- PASO 1: Identificar y extraer valores únicos de regiones ---
regiones_unificadas = pd.concat([
    m_seasons['RegionW'],
    m_seasons['RegionX'],
    m_seasons['RegionY'],
    m_seasons['RegionZ']
]).unique()

# --- PASO 2: Crear la tabla de referencia 'Regiones' ---
df_regiones = pd.DataFrame({
    'region_nombre': regiones_unificadas
}).reset_index()

# Ajustamos el ID para que empiece en 1
df_regiones['region_id'] = df_regiones['index'] + 1
df_regiones = df_regiones[['region_id', 'region_nombre']]

# --- PASO 3: Sustituir nombres por IDs en la tabla original ---
mapa_regiones = dict(zip(df_regiones['region_nombre'], df_regiones['region_id']))

m_seasons_normalizada = m_seasons.copy()
m_seasons_normalizada['RegionW'] = m_seasons_normalizada['RegionW'].map(mapa_regiones)
m_seasons_normalizada['RegionX'] = m_seasons_normalizada['RegionX'].map(mapa_regiones)
m_seasons_normalizada['RegionY'] = m_seasons_normalizada['RegionY'].map(mapa_regiones)
m_seasons_normalizada['RegionZ'] = m_seasons_normalizada['RegionZ'].map(mapa_regiones)

# Mostrar resultados
print("--- Nueva Tabla: Regiones ---")
print(df_regiones.head())

print("\n--- Tabla MSeasons (Con los ID de las regiones) ---")
print(m_seasons_normalizada[['Season', 'RegionW', 'RegionX', 'RegionY', 'RegionZ']].head())

   Season     DayZero RegionW    RegionX    RegionY    RegionZ
0    1985  10/29/1984    East       West    Midwest  Southeast
1    1986  10/28/1985    East    Midwest  Southeast       West
2    1987  10/27/1986    East  Southeast    Midwest       West
3    1988  11/02/1987    East    Midwest  Southeast       West
4    1989  10/31/1988    East       West    Midwest  Southeast
/
--- Nueva Tabla: Regiones ---
   region_id region_nombre
0          1          East
1          2       Atlanta
2          3   Albuquerque
3          4           NA1
4          5          TBD1

--- Tabla MSeasons (Con los ID de las regiones) ---
   Season  RegionW  RegionX  RegionY  RegionZ
0    1985        1        6        7        8
1    1986        1        7        8        6
2    1987        1        8        7        6
3    1988        1        7        8        6
4    1989        1        6        7        8


Tabla 2

In [4]:
# TABLA 2 PARA NORMAIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/WSeasons.csv"
response = requests.get(url)
response.raise_for_status()

# 1.Cargar la tabla
w_seasons = pd.read_csv(io.StringIO(response.text))

# 2. Extraer nombres únicos de regiones (W, X, Y, Z)
nombres_regiones_w = pd.unique(w_seasons[['RegionW', 'RegionX', 'RegionY', 'RegionZ']].values.ravel('K'))

# 3. Crear la tabla maestra 'W_Regiones'
df_regiones_w = pd.DataFrame({
    'region_nombre': nombres_regiones_w
}).reset_index()

df_regiones_w['region_id'] = df_regiones_w['index'] + 1
df_regiones_w = df_regiones_w[['region_id', 'region_nombre']]

# 4. Sustituir los nombres por IDs en WSeasons
mapa_regiones_w = dict(zip(df_regiones_w['region_nombre'], df_regiones_w['region_id']))

w_seasons_norm = w_seasons.copy()
columnas_region = ['RegionW', 'RegionX', 'RegionY', 'RegionZ']

for col in columnas_region:
    w_seasons_norm[col] = w_seasons_norm[col].map(mapa_regiones_w)

# --- RESULTADOS ---
print("--- Tabla: Regiones Femeninas ---")
print(df_regiones_w.head())

print("\n--- Tabla WSeasons Normalizada ---")
print(w_seasons_norm[['Season', 'RegionW', 'RegionX', 'RegionY', 'RegionZ']].head())

--- Tabla: Regiones Femeninas ---
   region_id region_nombre
0          1          East
1          2   Chattanooga
2          3   Albuquerque
3          4        Dallas
4          5    Greensboro

--- Tabla WSeasons Normalizada ---
   Season  RegionW  RegionX  RegionY  RegionZ
0    1998        1       18       19       20
1    1999        1       19       18       20
2    2000        1       18       19       20
3    2001        1       18       19       20
4    2002        1       20       19       18


Tabla 3

In [5]:
# TABLA 3 PARA NORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeamCoaches.csv"
response = requests.get(url)
response.raise_for_status()

# 1.Cargar la tabla
m_coaches = pd.read_csv(io.StringIO(response.text))

# 2. Crear la tabla maestra 'Entrenadores' con nombres y ID unicos
entrenadores_unicos = m_coaches[['CoachName']].drop_duplicates().reset_index(drop=True)
entrenadores_unicos['coach_id'] = entrenadores_unicos.index + 1

# 3. Normalizar la tabla original, remplazar el texto por el ID
m_team_coaches_norm = m_coaches.merge(entrenadores_unicos, on='CoachName')
m_team_coaches_norm = m_team_coaches_norm.drop(columns=['CoachName'])

# Reordenamos para que se vea mejor (ID de entrenador primero)
m_team_coaches_norm = m_team_coaches_norm[['Season', 'TeamID', 'coach_id', 'FirstDayNum', 'LastDayNum']]

# --- RESULTADOS ---
print("--- Nueva Tabla: Entrenadores ---")
print(entrenadores_unicos.head())

print("\n--- Tabla MTeamCoaches Normalizada ---")
print(m_team_coaches_norm.head())

--- Nueva Tabla: Entrenadores ---
        CoachName  coach_id
0   reggie_minton         1
1     bob_huggins         2
2  wimp_sanderson         3
3    james_oliver         4
4   davey_whitney         5

--- Tabla MTeamCoaches Normalizada ---
   Season  TeamID  coach_id  FirstDayNum  LastDayNum
0    1985    1102         1            0         154
1    1985    1103         2            0         154
2    1985    1104         3            0         154
3    1985    1106         4            0         154
4    1985    1108         5            0         154


Tabla 4

In [6]:
# TABLA 4 PARA NORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/Conferences.csv"
response = requests.get(url)
response.raise_for_status()

# 1.Cargar la tabla
conferences = pd.read_csv(io.StringIO(response.text))

# 2. Crear la tabla Maestra con un ID numérico (PK)
conferences_norm = conferences.copy()
conferences_norm.insert(0, 'conf_id', range(1, len(conferences_norm) + 1))

# 3. Limpieza de datos
conferences_norm['Description'] = conferences_norm['Description'].str.strip()
conferences_norm['ConfAbbrev'] = conferences_norm['ConfAbbrev'].str.upper()

# --- RESULTADOS ---
print("--- Tabla Conferences Normalizada ---")
# Mostramos el ID numérico, la abreviatura y el nombre completo
print(conferences_norm.head())

# Guardar mapeo para usarlo en futuras normalizaciones de tablas de equipos
mapa_conf_id = dict(zip(conferences_norm['ConfAbbrev'], conferences_norm['conf_id']))

--- Tabla Conferences Normalizada ---
   conf_id ConfAbbrev                   Description
0        1      A_SUN       Atlantic Sun Conference
1        2      A_TEN        Atlantic 10 Conference
2        3        AAC  American Athletic Conference
3        4        ACC     Atlantic Coast Conference
4        5        AEC       America East Conference


Tabla 5

In [7]:
# TABLA 5 PARA NORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeams.csv"
response = requests.get(url)
response.raise_for_status()

# 1.Cargar la tabla
m_teams = pd.read_csv(io.StringIO(response.text))

# 2. Limpieza de nombres (Normalización de Strings)
m_teams['TeamName'] = m_teams['TeamName'].str.strip().str.title()

# 3. Verificación de unicidad
# Nos aseguramos de que no existan TeamID duplicados antes de establecerla como PK
if not m_teams['TeamID'].is_unique:
    m_teams = m_teams.drop_duplicates(subset=['TeamID'])

# 4. Optimización de tipos de datos, cambiamos el tipo de dato de la fecha para mayor eficiencia en la busqueda
m_teams['FirstD1Season'] = m_teams['FirstD1Season'].astype(int)
m_teams['LastD1Season'] = m_teams['LastD1Season'].astype(int)

# --- RESULTADOS ---
print("--- Tabla MTeams Normalizada (Entidad Maestra) ---")
print(m_teams.head())

# Esta tabla ahora servirá como el catálogo principal para todos los procesos de desnormalización

--- Tabla MTeams Normalizada (Entidad Maestra) ---
   TeamID     TeamName  FirstD1Season  LastD1Season
0    1101  Abilene Chr           2014          2026
1    1102    Air Force           1985          2026
2    1103        Akron           1985          2026
3    1104      Alabama           1985          2026
4    1105  Alabama A&M           2000          2026


PROCESO DE DESNORMALIZACION

Tabla 1

In [8]:
# TABLA 1 PARA DESNORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MGameCities.csv"
response = requests.get(url)
response.raise_for_status()

url2 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/Cities.csv"
response2 = requests.get(url2)
response2.raise_for_status()

# 1.Cargar la tabla
m_game_cities = pd.read_csv(io.StringIO(response.text))
cities = pd.read_csv(io.StringIO(response2.text))

# 2. Unir (Merge) las tablas para obtener nombres de ciudades y estados
df_juegos_ciudades = pd.merge(m_game_cities, cities, on='CityID', how='left')

# 3. Limpieza post-unión
# Reordenamos las columnas para que la lectura sea más natural
df_juegos_ciudades = df_juegos_ciudades[['Season', 'DayNum', 'WTeamID', 'LTeamID', 'City', 'State', 'CRType']]

# --- RESULTADOS ---
print("--- Datos Desnormalizados: Juegos y sus Ciudades ---")
# Ahora en una sola fila vemos el ID del equipo y el nombre real de la ciudad
print(df_juegos_ciudades.head())

--- Datos Desnormalizados: Juegos y sus Ciudades ---
   Season  DayNum  WTeamID  LTeamID         City State   CRType
0    2010       7     1143     1293     Berkeley    CA  Regular
1    2010       7     1314     1198  Chapel Hill    NC  Regular
2    2010       7     1326     1108     Columbus    OH  Regular
3    2010       7     1393     1107     Syracuse    NY  Regular
4    2010       9     1143     1178     Berkeley    CA  Regular


Tabla 2

In [9]:
# TABLA 2 PARA DESNORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/WGameCities.csv"
response = requests.get(url)
response.raise_for_status()

url2 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/Cities.csv"
response2 = requests.get(url2)
response2.raise_for_status()

# 1.Cargar la tabla
w_game_cities = pd.read_csv(io.StringIO(response.text))
cities = pd.read_csv(io.StringIO(response2.text))

# 2. Unir las tablas usando 'CityID' como llave
# Esto trae el nombre de la ciudad y el estado a la tabla de juegos
df_juegos_mujeres_completo = pd.merge(w_game_cities, cities, on='CityID', how='left')

# 3. Refinar la estructura, eliminar el ID de la city ya que unimos el nombre
df_juegos_mujeres_completo = df_juegos_mujeres_completo[['Season', 'DayNum', 'WTeamID', 'LTeamID', 'City', 'State']]

# --- RESULTADOS ---
print("--- Datos Desnormalizados: Juegos Femeninos y Ubicaciones ---")
print(df_juegos_mujeres_completo.head())

--- Datos Desnormalizados: Juegos Femeninos y Ubicaciones ---
   Season  DayNum  WTeamID  LTeamID            City State
0    2010      11     3103     3237           Akron    OH
1    2010      11     3104     3399  Corpus Christi    TX
2    2010      11     3110     3224      Washington    DC
3    2010      11     3111     3267      Huntington    WV
4    2010      11     3119     3447      West Point    NY


Tabla 3

In [10]:
# TABLA 3 PARA DESNORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeamConferences.csv"
response = requests.get(url)
response.raise_for_status()

url2 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeams.csv"
response2 = requests.get(url2)
response2.raise_for_status()

url3 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/Conferences.csv"
response3 = requests.get(url3)
response3.raise_for_status()

# 1.Cargar la tabla
m_team_conf = pd.read_csv(io.StringIO(response.text))
m_teams = pd.read_csv(io.StringIO(response2.text))
conferences = pd.read_csv(io.StringIO(response3.text))

# 2. Primera unión: Traer el nombre del equipo
df_paso1 = pd.merge(m_team_conf, m_teams[['TeamID', 'TeamName']], on='TeamID', how='left')

# 3. Segunda unión: Traer la descripción completa de la conferencia
df_final_conf = pd.merge(df_paso1, conferences, on='ConfAbbrev', how='left')

# 4. Organizar para el reporte
df_final_conf = df_final_conf[['Season', 'TeamName', 'Description', 'ConfAbbrev']]
df_final_conf.rename(columns={'Description': 'ConferenceName'}, inplace=True)

# --- RESULTADOS ---
print("--- Vista Desnormalizada: Equipos y sus Conferencias ---")
print(df_final_conf.head(10))

--- Vista Desnormalizada: Equipos y sus Conferencias ---
   Season        TeamName                                ConferenceName  \
0    1985       Air Force                   Western Athletic Conference   
1    1985           Akron                        Ohio Valley Conference   
2    1985         Alabama                       Southeastern Conference   
3    1985      Alabama St                 Southwest Athletic Conference   
4    1985       Alcorn St                 Southwest Athletic Conference   
5    1985    Alliant Intl                                   Independent   
6    1985   American Univ  Eastern Collegiate Athletic Conference South   
7    1985  Appalachian St                           Southern Conference   
8    1985         Arizona                         Pacific-10 Conference   
9    1985      Arizona St                         Pacific-10 Conference   

  ConfAbbrev  
0        wac  
1        ovc  
2        sec  
3       swac  
4       swac  
5        ind  
6      ecacs

Tabla 4

In [11]:
# TABLA 4 PARA DESNORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeamCoaches.csv"
response = requests.get(url)
response.raise_for_status()

url2 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeams.csv"
response2 = requests.get(url2)
response2.raise_for_status()

# 1. Cargar las tablas necesarias
m_coaches = pd.read_csv(io.StringIO(response.text))
m_teams = pd.read_csv(io.StringIO(response2.text))

# 2. Unir para traer el nombre del Equipo (TeamName)
# Usamos 'TeamID' como llave
df_coaches_desnorm = pd.merge(m_coaches, m_teams[['TeamID', 'TeamName']], on='TeamID', how='left')

# 3. Organizar las columnas para que sea un reporte de gestión
# Queremos ver la temporada, el equipo, el entrenador y cuánto tiempo estuvo (días)
df_reporte_coaches = df_coaches_desnorm[[
    'Season',
    'TeamName',
    'CoachName',
    'FirstDayNum',
    'LastDayNum'
]]

# 4. Ordenar por temporada y equipo para facilitar la lectura
df_reporte_coaches = df_reporte_coaches.sort_values(by=['Season', 'TeamName'])

# --- RESULTADOS ---
print("--- Reporte Desnormalizado de Entrenadores por Equipo ---")
print(df_reporte_coaches.head(10))

--- Reporte Desnormalizado de Entrenadores por Equipo ---
   Season        TeamName       CoachName  FirstDayNum  LastDayNum
0    1985       Air Force   reggie_minton            0         154
1    1985           Akron     bob_huggins            0         154
2    1985         Alabama  wimp_sanderson            0         154
3    1985      Alabama St    james_oliver            0         154
4    1985       Alcorn St   davey_whitney            0         154
5    1985    Alliant Intl    freddie_goss            0         154
6    1985   American Univ     ed_tapscott            0         154
7    1985  Appalachian St  kevin_cantwell            0         154
8    1985         Arizona      lute_olson            0         154
9    1985      Arizona St   bob_weinhauer            0         154


Tabla 5

In [12]:
# TABLA 5 PARA DESNORMALIZAR
# 1. Datos Desnormalizados
url = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeamConferences.csv"
response = requests.get(url)
response.raise_for_status()

url2 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MSeasons.csv"
response2 = requests.get(url2)
response2.raise_for_status()

url3 = "https://raw.githubusercontent.com/SandraFB/IngenieriaDatos/refs/heads/main/MTeams.csv"
response3 = requests.get(url3)
response3.raise_for_status()

# 1.Cargar la tabla
m_team_conf = pd.read_csv(io.StringIO(response.text))
m_seasons = pd.read_csv(io.StringIO(response2.text))
m_teams = pd.read_csv(io.StringIO(response3.text))

# 2. Unir el equipo con su conferencia y nombre del equipo
df_paso1 = pd.merge(m_team_conf, m_teams[['TeamID', 'TeamName']], on='TeamID')

# 3. Desnormalizar con la tabla de temporadas para traer contexto regional Esto nos permite ver en una sola tabla el equipo y las regiones de ese año
df_final_historico = pd.merge(
    df_paso1,
    m_seasons[['Season', 'RegionW', 'RegionX', 'RegionY', 'RegionZ']],
    on='Season'
)

# 4. Seleccionar columnas clave para un reporte de analítica
df_final_historico = df_final_historico[[
    'Season', 'TeamName', 'ConfAbbrev', 'RegionW', 'RegionX'
]]

# --- RESULTADOS ---
print("--- Vista Desnormalizada de la tabla 5: Evolución de Equipos y Regiones ---")
print(df_final_historico.head())

--- Vista Desnormalizada de la tabla 5: Evolución de Equipos y Regiones ---
   Season    TeamName ConfAbbrev RegionW RegionX
0    1985   Air Force        wac    East    West
1    1985       Akron        ovc    East    West
2    1985     Alabama        sec    East    West
3    1985  Alabama St       swac    East    West
4    1985   Alcorn St       swac    East    West
